# 20 - Python 3.11 New Features


## 1. `ExceptionGroup` & `except*` Syntax

Python 3.11 ile tanıtılan `ExceptionGroup` (ve temel istisnalar için `BaseExceptionGroup`), eşzamanlı işlemler veya toplu doğrulamalar sırasında aynı anda fırlatılan birden fazla bağımsız istisnayı tek bir yapısal hata grubu altında toplamayı ve `except*` sözdizimiyle türlerine göre ayrı ayrı yakalamayı sağlar[cite: 112].

**Syntax**

```Python
eg = ExceptionGroup('Batch errors', [FileNotFoundError('a.png'), KeyError('user')])

try:
    raise eg
except* FileNotFoundError as fnf_group:
    print(fnf_group.exceptions)
except* KeyError as key_group:
    print(key_group.exceptions)
```

**Temel Kurallar ve Kullanım Detayları**

* **`except*` (Exception Group Matching):** Klasik `except` bloğundan farklı olarak `except*`, eşleşen tüm alt istisnaları ayıklayıp yeni bir `ExceptionGroup` örneği olarak yakalar ve bir `try` bloğu içindeki birden fazla `except*` dalı aynı hata grubu üzerinde sırayla çalışabilir.
* **Yakalanmayan İstisnaların Yayılması:** Grup içindeki alt istisnalardan biri hiçbir `except*` bloğuyla eşleşmezse, kalan istisnaları içeren yeni bir `ExceptionGroup` otomatik olarak yukarıya (çağıran koda) fırlatılır.
* **İç İçe (Nested) Gruplar ve `.subgroup()` / `.split()`:** `ExceptionGroup` içine başka bir `ExceptionGroup` yerleştirilebilir; `except*` eşleşen yaprak istisnaları iç içe hiyerarşiyi koruyarak otomatik ayıklar ve `.exceptions` demeti üzerinden erişime açar[cite: 112].


In [ ]:
eg = ExceptionGroup(
    'Custom exception message',
    [
        FileNotFoundError('file_a.png does not exist'),
        FileNotFoundError('file_b.png does not exist'),
        KeyError('No user found in dict'),
        ExceptionGroup('Nested group', [ValueError('Invalid payload')]),
    ],
)

try:
    raise eg
except* FileNotFoundError as fnf_eg:
    for err in fnf_eg.exceptions:
        print('Handled FileNotFoundError:', err)
except* KeyError as key_eg:
    for err in key_eg.exceptions:
        print('Handled KeyError:', err)
except* ValueError as val_eg:
    for err in val_eg.exceptions:
        print('Handled Nested ValueError:', err)


## 2. Structured Concurrency with `asyncio.TaskGroup`

`asyncio.TaskGroup`, Python 3.11'de `asyncio.gather()` fonksiyonuna modern ve güvenli bir alternatif olarak eklenen eşzamanlı görev yöneticisidir; `async with` bloğundan çıkılmadan önce gruptaki tüm görevlerin tamamlanmasını garanti eder ve hata durumunda diğer görevleri otomatik iptal ederek hataları `ExceptionGroup` olarak fırlatır[cite: 110].

**Syntax**

```Python
try:
    async with asyncio.TaskGroup() as tg:
        task_1 = tg.create_task(coro_a())
        task_2 = tg.create_task(coro_b())
    print(task_1.result(), task_2.result())
except* ValueError as eg:
    print(eg.exceptions)
```

**`TaskGroup` vs. `asyncio.gather()` Farkları**

* **Otomatik İptal ve Temizlik:** `TaskGroup` içindeki görevlerden biri hata fırlattığında, gruptaki diğer aktif görevler otomatik olarak iptal (`cancel()`) edilir; `gather()` ise varsayılan olarak diğer görevleri iptal etmeden arka planda bırakır.
* **Doğrudan `.result()` Erişimi:** `async with` bloğu hatasız tamamlandığında tüm görevlerin bittiği kesin olduğundan, ek `await` gerektirmeden doğrudan `task.result()` ile dönüş değerleri okunabilir.
* **Yapısal Hata Toplama:** Eşzamanlı başlayan birden fazla görev hata üretirse, tüm yakalanan istisnalar tek bir `ExceptionGroup` içinde birleştirilip `except*` ile yönetilir[cite: 110].


In [ ]:
import asyncio


async def fetch_data(item_id: int) -> str:
    if item_id == 0:
        raise ValueError('No data available for ID 0')
    await asyncio.sleep(0.05 * item_id)
    return f'Data-{item_id}'


async def read_missing_file(filename: str) -> str:
    await asyncio.sleep(0.02)
    raise FileNotFoundError(f'{filename} not found')


async def run_success_group() -> None:
    async with asyncio.TaskGroup() as tg:
        t1: asyncio.Task[str] = tg.create_task(fetch_data(1))
        t2: asyncio.Task[str] = tg.create_task(fetch_data(2))
    print('Success results:', t1.result(), t2.result())


async def run_failing_group() -> None:
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(fetch_data(1))
            tg.create_task(fetch_data(0))
            tg.create_task(read_missing_file('123.png'))
    except* ValueError as val_eg:
        print('Caught ValueError group:', val_eg.exceptions)
    except* FileNotFoundError as fnf_eg:
        print('Caught FileNotFoundError group:', fnf_eg.exceptions)


await run_success_group()
await run_failing_group()


## 3. The `typing.Self` Type Annotation

Python 3.11 (PEP 673) ile `typing` modülüne eklenen `Self` tipi, bir sınıfın kendi örneğini döndüren metotlarda (zincirleme/fluent API metotları, `@classmethod` alternatif yapıcıları ve `__enter__` bağlam yöneticileri) alt sınıfların tip bilgisini kaybetmeden doğru dönüş tipi belirtmeyi sağlar[cite: 111].

**Syntax**

```python
from typing import Self


class Shape:
    def set_scale(self, scale: float) -> Self:
        self.scale = scale
        return self
```

**Neden `Self` Kullanılır?**

* **Kalıtımda Doğru Tip Çıkarımı:** Üst sınıftaki `set_scale(self) -> Shape` metodu alt sınıf olan `Circle` üzerinden çağrıldığında statik tip denetleyiciler dönüş tipini `Shape` sanır ve `Circle`'a özgü `set_radius()` metodunu tanımaz; dönüş tipi `-> Self` yapıldığında ise `Circle().set_scale(0.5).set_radius(10)` zinciri hatasız tanınır.
* **Alternatif Yapıcılar (`@classmethod`):** Sınıf adı henüz tanımlanırken ileri referans (`'Fruit'`) veya `TypeVar` karmaşasına gerek kalmadan `@classmethod` dönüş tipi doğrudan `-> Self` olarak yazılır.
* **Bağlam Yöneticileri (`__enter__`):** `def __enter__(self) -> Self:` tanımı, `with` bloğundaki `as` değişkeninin alt sınıf tipini korumasını sağlar[cite: 111].


In [ ]:
from typing import Self


class Shape:
    def __init__(self) -> None:
        self.scale: float = 1.0

    def set_scale(self, scale: float) -> Self:
        self.scale = scale
        return self


class Circle(Shape):
    def __init__(self) -> None:
        super().__init__()
        self.radius: float = 1.0

    def set_radius(self, radius: float) -> Self:
        self.radius = radius
        return self


class Fruit:
    def __init__(self, name: str) -> None:
        self.name = name

    @classmethod
    def create_super_fruit(cls, name: str, power: int) -> Self:
        super_name: str = f'{name}{"!" * power}'
        return cls(super_name)

    def __enter__(self) -> Self:
        return self

    def __exit__(self, exc_type: object, exc_val: object, exc_tb: object) -> None:
        pass


circle = Circle().set_scale(0.5).set_radius(10.0)
print(f'Circle -> scale: {circle.scale}, radius: {circle.radius}')

with Fruit.create_super_fruit('Banana', 5) as super_banana:
    print('Super fruit:', super_banana.name)
